# 03 — Baseline và Isolation Forest trên validation

Bước 6, chưa triển khai mô hình phân lớp. Chạy bốn cấu hình đã định trước trong configs/isolation.json; fit không truyền nhãn trên train, chọn theo Average Precision của validation. Chỉ chọn trên seed 42; seed 43/44 đo dao động với cấu hình đã chọn. Đây là lựa chọn có hỗ trợ nhãn validation, không phải thí nghiệm hoàn toàn không nhãn từ đầu đến cuối. Không nạp test.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
while not (ROOT / "configs/isolation.json").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Mở notebook trong repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from maintenance.models import run_isolation
manifest = run_isolation(ROOT)
assert manifest["test_evaluated"] is False
print("Đã chạy baseline/Isolation Forest; chưa mở test.")

Đã chạy baseline/Isolation Forest; chưa mở test.


In [2]:
candidates = pd.read_csv(ROOT / "reports/tables/isolation/candidates.csv")
validation = pd.read_csv(ROOT / "reports/tables/isolation/validation.csv")
display(candidates[["split_id", "candidate_id", "average_precision", "precision_at_50"]])
display(validation.loc[validation.seed.eq(42), ["split_id", "model_family", "average_precision", "roc_auc", "tp", "fp", "fn", "tn", "precision", "recall", "f1", "precision_at_50", "recall_at_50"]])

,split_id,candidate_id,average_precision,precision_at_50
0,ordered-gap100-v1,if-base-m256,0.110566,0.12
1,ordered-gap100-v1,if-base-m1024,0.161526,0.14
2,ordered-gap100-v1,if-physical-m256,0.204530,0.20
3,ordered-gap100-v1,if-physical-m1024,0.320247,0.30
4,stratified-seed42-v1,if-base-m256,0.102145,0.16
5,stratified-seed42-v1,if-base-m1024,0.140648,0.18
6,stratified-seed42-v1,if-physical-m256,0.226655,0.38
7,stratified-seed42-v1,if-physical-m1024,0.258498,0.40


,split_id,model_family,average_precision,roc_auc,tp,fp,fn,tn,precision,recall,f1,precision_at_50,recall_at_50
0,ordered-gap100-v1,isolation_forest,0.320247,0.859300,15,31,28,1826,0.326087,0.348837,0.337079,0.3,0.348837
1,ordered-gap100-v1,random,0.019587,0.454835,43,1747,0,110,0.024022,1.000000,0.046918,0.0,0.000000
2,ordered-gap100-v1,always_normal,0.022632,0.500000,0,0,43,1857,NaN,0.000000,0.000000,NaN,NaN
9,stratified-seed42-v1,isolation_forest,0.258498,0.842346,20,25,48,1907,0.444444,0.294118,0.353982,0.4,0.294118
10,stratified-seed42-v1,random,0.039845,0.575676,28,485,40,1447,0.054581,0.411765,0.096386,0.0,0.000000
11,stratified-seed42-v1,always_normal,0.034000,0.500000,0,0,68,1932,NaN,0.000000,0.000000,NaN,NaN


## Diễn giải và giới hạn

Average Precision chọn cấu hình, không phải accuracy. Ngưỡng F1 cũng chọn trên validation; chỉ số tại ngưỡng không phải kết quả độc lập. Điểm lớn hơn = bất thường hơn, không phải xác suất hỏng. Baseline ngẫu nhiên sử dụng hash khóa/seed, không nhìn nhãn; baseline luôn bình thường không có xếp hạng có ích nên không báo top-k cho nó. Ô thiếu ở chỉ số không xác định/không áp dụng giữ trống, không tự đổi thành 0.

Xem docs/ISOLATION_FOREST.md và bảng seed_summary để biết độ dao động; ba seed không phải ba tập dữ liệu độc lập. Danh sách ưu tiên và mô hình lưu local, không nhúng bản ghi nguồn vào notebook.

In [3]:
summary = pd.read_csv(ROOT / "reports/tables/isolation/seed_summary.csv", header=[0, 1], index_col=[0, 1])
display(summary)
for selection in manifest["selections"]:
    print(selection["split_id"], selection["candidate_id"])
print("Chưa chạy họ phụ phân lớp; test giữ cho bước đánh giá cuối.")

average_precision             roc_auc  \
                                                   mean       std      mean   
split_id             model_family                                             
ordered-gap100-v1    always_normal             0.022632  0.000000  0.500000   
                     isolation_forest          0.259280  0.053104  0.852997   
                     random                    0.018998  0.000685  0.435307   
stratified-seed42-v1 always_normal             0.034000  0.000000  0.500000   
                     isolation_forest          0.249566  0.013079  0.839118   
                     random                    0.035321  0.003942  0.529237   

                                                precision_at_50            \
                                            std            mean       std   
split_id             model_family                                           
ordered-gap100-v1    always_normal     0.000000             NaN       NaN   
                     isolation_forest  0.009512        0.266667  0.030551   
                     random            0.022205        0.000000  0.000000   
stratified-seed42-v1 always_normal     0.000000             NaN       NaN   
                     isolation_forest  0.008950        0.373333  0.023094   
                     random            0.040440        0.000000  0.000000   

                                      recall_at_50                  f1  \
                                              mean       std      mean   
split_id             model_family                                        
ordered-gap100-v1    always_normal             NaN       NaN  0.000000   
                     isolation_forest     0.310078  0.035524  0.317121   
                     random               0.000000  0.000000  0.046262   
stratified-seed42-v1 always_normal             NaN       NaN  0.000000   
                     isolation_forest     0.274510  0.016981  0.334844   
                     random               0.000000  0.000000  0.083896   

                                                 
                                            std  
split_id             model_family                
ordered-gap100-v1    always_normal     0.000000  
                     isolation_forest  0.018701  
                     random            0.000759  
stratified-seed42-v1 always_normal     0.000000  
                     isolation_forest  0.024163  
                     random            0.011204

ordered-gap100-v1 if-physical-m1024
stratified-seed42-v1 if-physical-m1024
Chưa chạy họ phụ phân lớp; test giữ cho bước đánh giá cuối.
